# 01 — Data Exploration (EDA)
**SmartBiz AI V1.0 — Phase 3: AI Research Pipeline**

This notebook performs exploratory data analysis on the sample culinary MSME sales dataset.

### Objectives
1. Understand the dataset structure and basic statistics.
2. Visualize sales trends over time.
3. Analyze sales quantity distribution.
4. Identify temporal patterns (day-of-week, promotion effects).

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['figure.dpi'] = 100

print('Libraries loaded.')

## 1. Load Dataset

In [ ]:
from src.data.loader import load_csv

df = load_csv('../data/raw/sample_sales_data.csv')
df['date'] = pd.to_datetime(df['date'])
print(f'Shape: {df.shape}')
df.head(10)

## 2. Basic Statistics

In [ ]:
print('--- Data Types ---')
print(df.dtypes)
print('\n--- Descriptive Statistics ---')
df.describe()

In [ ]:
print('--- Missing Values ---')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

## 3. Sales Trend Visualization

In [ ]:
daily_total = df.groupby('date')['sales_qty'].sum().reset_index()

plt.figure(figsize=(14, 5))
plt.plot(daily_total['date'], daily_total['sales_qty'], marker='o', linewidth=1.5, markersize=4)
plt.title('Daily Total Sales Quantity', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Total Sales Qty')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/daily_sales_trend.png', dpi=150)
plt.show()

In [ ]:
# Sales trend per product
for pid in sorted(df['product_id'].unique()):
    product_df = df[df['product_id'] == pid]
    plt.plot(product_df['date'], product_df['sales_qty'], marker='.', label=f'Product {pid}')

plt.title('Sales Trend by Product', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Sales Qty')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/sales_trend_by_product.png', dpi=150)
plt.show()

## 4. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['sales_qty'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Sales Quantity Distribution')
axes[0].set_xlabel('Sales Qty')
axes[0].set_ylabel('Frequency')

axes[1].boxplot([df[df['product_id'] == pid]['sales_qty'] for pid in sorted(df['product_id'].unique())],
                labels=[f'P{pid}' for pid in sorted(df['product_id'].unique())])
axes[1].set_title('Sales Qty Boxplot by Product')
axes[1].set_ylabel('Sales Qty')

plt.tight_layout()
plt.savefig('../results/plots/distribution_analysis.png', dpi=150)
plt.show()

## 5. Temporal Pattern Analysis

In [ ]:
df['day_of_week'] = df['date'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

day_avg = df.groupby('day_of_week')['sales_qty'].mean().reindex(day_order)

plt.figure(figsize=(10, 5))
day_avg.plot(kind='bar', color='teal', edgecolor='black', alpha=0.8)
plt.title('Average Sales by Day of Week', fontsize=14)
plt.xlabel('Day of Week')
plt.ylabel('Average Sales Qty')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/sales_by_day_of_week.png', dpi=150)
plt.show()

In [ ]:
# Promotion effect
promo_avg = df.groupby('promotion')['sales_qty'].mean()

plt.figure(figsize=(6, 5))
promo_avg.plot(kind='bar', color=['#e74c3c', '#2ecc71'], edgecolor='black')
plt.title('Average Sales: Promotion vs No Promotion', fontsize=14)
plt.xlabel('Promotion (0=No, 1=Yes)')
plt.ylabel('Average Sales Qty')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/promotion_effect.png', dpi=150)
plt.show()

print(f'No promotion average: {promo_avg.get(0, 0):.1f}')
print(f'With promotion average: {promo_avg.get(1, 0):.1f}')

## 6. Correlation Matrix

In [ ]:
numeric_df = df[['sales_qty', 'price', 'promotion']].copy()
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.matshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
fig.colorbar(cax)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45)
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=12)
plt.title('Correlation Matrix', pad=20)
plt.tight_layout()
plt.savefig('../results/plots/correlation_matrix.png', dpi=150)
plt.show()

## Summary
- Dataset has **110 rows** across **5 products** and **22 days**.
- Clear **weekly seasonality** visible in daily totals.
- **Promotions** correlate with higher sales quantities.
- No missing values detected in the raw dataset.
- Data is clean and ready for the preprocessing pipeline.